In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

In [ ]:
%cd StyleTTS2

import torch
import argparse
from tqdm import tqdm
import cProfile
import os

import nltk
nltk.download('punkt_tab')

# Import your new specialized modules
from Adversarial_TTS.model_loader import initialize_environment, load_optimizer
from Adversarial_TTS.core_logic import run_optimization_generation
from Adversarial_TTS.reporting import finalize_run

In [ ]:
class NotebookArgs:
    def __init__(self):
        # String parameters
        self.ground_truth_text = "I think the NFL is lame and boring"
        self.target_text = "The Seattle Seahawks are the best Team in the world"

        # Numeric parameters
        self.loop_count = 1
        self.num_generations = 100
        self.pop_size = 200
        self.iv_scalar = 0.5
        self.size_per_phoneme = 1
        self.batch_size = 100

        # Boolean parameters
        self.notify = False
        self.subspace_optimization = False

        # Enum/Selection parameters
        self.mode = "TARGETED"
        self.ACTIVE_OBJECTIVES = ["PESQ", "WHISPER_PROB"]
        self.thresholds = ["PESQ=0.2", "WHISPER_PROB=0.6"]

args = NotebookArgs()

# 1. Set Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Initialize Environment
# This handles Enums, Thresholds, Model Loading, and Reference Data generation
config_data, model_data, audio_data, embedding_data = initialize_environment(args, device)

if config_data is None:
    print("Initialization failed.")
else:
    print(f"Starting Optimization Loop...")

In [ ]:
# 4. Main Loop

if torch.cuda.device_count() > 1:
    model_data.tts_model = torch.nn.DataParallel(model_data.tts_model)
    model_data.asr_model = torch.nn.DataParallel(model_data.asr_model)

for iteration in tqdm(range(config_data.loop_count), desc="Total Progress"):
    # Run the generation loop (Core Logic)
    fitness_data, progress_bar, stop_optimization, gen = run_optimization_generation(
        config_data, model_data, audio_data, embedding_data, iteration, device
    )

    # 5. Finalize and Save Results
    # This creates folders, saves audio, generates graphs, and sends notifications
    folder_path = finalize_run(config_data, fitness_data, model_data, audio_data, progress_bar, gen, device)

    model_data.optimizer = load_optimizer(audio_data, config_data)

In [ ]:
extract = False
if extract:
    !zip -r colab_files.zip /content/StyleTTS2/outputs/h_text/

    from google.colab import files
    files.download('colab_files.zip')